### Data Ingestion

In [8]:
###Document Structure

from langchain_core.documents import Document

In [9]:
from langchain_community.document_loaders import PyPDFLoader

# Diet charts — English-only (Hindi already removed)
diet_loader = PyPDFLoader("../data/DietCharts_Ayurveda_cleaned.pdf")
diet_pages = diet_loader.load()

# Ayurvedic product combinations mapped to diseases
product_loader = PyPDFLoader("../data/product_combination.pdf")
product_pages = product_loader.load()

print(f"Diet pages: {len(diet_pages)}, Product pages: {len(product_pages)}")

Diet pages: 582, Product pages: 6


In [ ]:
import uuid
from sentence_transformers import SentenceTransformer
import chromadb

### Chunking + Metadata Extraction

In [ ]:
import re

MEAL_SLOTS = [
    "Early Morning", "Breakfast", "Mid-Morning", "Lunch",
    "Evening Snack", "Evening", "Dinner", "Before Bed"
]
MEAL_PATTERN = re.compile(
    r'(?=(?:' + '|'.join(re.escape(s) for s in MEAL_SLOTS) + r'))', re.I
)

def extract_diet_metadata(text: str) -> dict:
    meta = {}

    chart = re.search(r'DIET CHART\s+(\d+)[:\s]+(.+?)(?:\n|FOR BMI|$)', text, re.I)
    if chart:
        meta['diet_chart_id'] = f"CHART_{chart.group(1).strip()}"
        meta['disease_condition'] = chart.group(2).strip().rstrip(',').strip()

    bmi = re.search(
        r'FOR BMI\s*([\d.]+\s*[-–]\s*[\d.]+|[><]\s*\d+|\d+\s*[-–]\s*\d+)',
        text, re.I
    )
    if bmi:
        meta['bmi_range'] = bmi.group(1).strip()

    gender = re.search(r'\b(MALE|FEMALE)\b', text, re.I)
    if gender:
        meta['gender'] = gender.group(1).capitalize()

    if 'diet_chart_id' not in meta:
        section = re.match(r'^([A-Z][^\n]{3,60})\n', text.strip())
        if section:
            meta['disease_condition'] = section.group(1).strip()

    ayur = re.findall(
        r'\b(Kapha|Pitta|Vata|Medohara|Rasayana|Lekhana|Ama-pachana|Tridosha|Agni)\b', text
    )
    if ayur:
        meta['ayurvedic_actions'] = list(set(ayur))
        for dosha in ('Kapha', 'Pitta', 'Vata'):
            if dosha in ayur:
                meta['dosha_target'] = dosha
                break

    return meta


def build_diet_chunks(pages: list) -> list:
    chunks, state = [], {}

    for page_doc in pages:
        text = (page_doc.page_content or "").strip()
        if not text:
            continue

        page_meta = extract_diet_metadata(text)

        for key in ('disease_condition', 'diet_chart_id', 'bmi_range', 'gender'):
            if key in page_meta:
                state[key] = page_meta[key]

        base_meta = {
            **state,
            **{k: v for k, v in page_meta.items()
               if k not in ('disease_condition', 'diet_chart_id', 'bmi_range', 'gender')},
            'page_number': page_doc.metadata.get('page', 0) + 1,
            'source': page_doc.metadata.get('source', ''),
            'doc_type': 'diet_chart',
        }

        parts = MEAL_PATTERN.split(text)
        for part in parts:
            part = part.strip()
            if not part:
                continue
            slot_match = re.match(
                r'^(' + '|'.join(re.escape(s) for s in MEAL_SLOTS) + r')', part, re.I
            )
            chunks.append(Document(
                page_content=part,
                metadata={
                    **base_meta,
                    'meal_slot': slot_match.group(1).title() if slot_match else 'General'
                }
            ))

    return chunks


diet_chunks = build_diet_chunks(diet_pages)
print(f"Diet chunks: {len(diet_chunks)}")

In [ ]:
def build_product_chunks(pages: list) -> list:
    full_text = '\n'.join(p.page_content for p in pages if p.page_content)
    entries = re.split(r'\n(?=[A-Z][A-Za-z\s,&+/\-]{3,80}\n)', full_text)

    chunks = []
    for i, entry in enumerate(entries):
        entry = entry.strip()
        if not entry:
            continue

        first_line = entry.split('\n')[0].strip()
        diseases = [d.strip() for d in re.split(r'\s*[+&]\s*', first_line) if d.strip()]

        products = re.findall(
            r'[\w\s]+ (?:tablet|tea|syrup|drops|kashaya|vati|churna|arishta|capsule)',
            entry, re.I
        )
        statuses = re.findall(r'\(Ready\)|\(Yet to manufacture\)', entry, re.I)
        classical = re.findall(
            r'[\w\s]+ (?:Vati|Kashaya|Churna|Arishta|Bhasma|Ghrita|Taila)',
            entry
        )

        meta = {
            'doc_type': 'product_recommendation',
            'disease_condition': ' + '.join(diseases),
            'is_combination': str(len(diseases) > 1),
            'products': ', '.join(set(p.strip() for p in products)),
            'classical_medications': ', '.join(set(c.strip() for c in classical)),
            'product_statuses': ', '.join(set(s.strip('()') for s in statuses)),
            'page_number': pages[min(i, len(pages) - 1)].metadata.get('page', 0) + 1,
            'source': pages[0].metadata.get('source', ''),
        }
        chunks.append(Document(page_content=entry, metadata=meta))

    return chunks


product_chunks = build_product_chunks(product_pages)
print(f"Product chunks: {len(product_chunks)}")

### Embedding and VectorStoreDB

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')

# PersistentClient avoids re-embedding on every notebook run
client = chromadb.PersistentClient(path="../chroma_db")
collection = client.get_or_create_collection(
    name="ayurveda_knowledge_base",
    metadata={"hnsw:space": "cosine"}
)

all_chunks = diet_chunks + product_chunks
print(f"Total chunks to embed: {len(all_chunks)}")

BATCH_SIZE = 64
for i in range(0, len(all_chunks), BATCH_SIZE):
    batch = all_chunks[i:i + BATCH_SIZE]
    texts = [c.page_content for c in batch]
    embeddings = embedding_model.encode(texts, show_progress_bar=False).tolist()
    # ChromaDB requires scalar metadata values — convert any lists to comma-joined strings
    metadatas = [
        {k: (', '.join(v) if isinstance(v, list) else v)
         for k, v in c.metadata.items()}
        for c in batch
    ]
    ids = [str(uuid.uuid4()) for _ in batch]
    collection.add(documents=texts, embeddings=embeddings, metadatas=metadatas, ids=ids)
    if (i // BATCH_SIZE) % 10 == 0:
        print(f"  Embedded batch {i // BATCH_SIZE + 1} / {len(all_chunks) // BATCH_SIZE + 1}")

print(f"\nTotal chunks stored in ChromaDB: {collection.count()}")